# Model 6: CF_Temp + CF_Count + Content + Pop + Graph RWR (Optuna Optimized)

**Precision@10:** 0.0560 | **Recall@10:** 0.2950

Our most advanced model adds two major innovations over Model 5:
1. **Graph Random Walk with Restart (RWR)** — discovers hidden connections by propagating signals across the full user-book network
2. **Optuna hyperparameter optimization** — automatically finds the best component weights using Bayesian search over 5-fold CV

## Step 1: Imports

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import csr_matrix, diags, bmat
import warnings
warnings.filterwarnings('ignore')


## Step 2: Load Data

In [ ]:
interactions = pd.read_csv('../data/interactions_train.csv').rename(
    columns={'u': 'user_id', 'i': 'book_id', 't': 'timestamp'})
books = pd.read_csv('../data/items.csv')

user_id_map = {o: n for n, o in enumerate(interactions['user_id'].unique())}
book_id_map = {o: n for n, o in enumerate(interactions['book_id'].unique())}
interactions['user_id'] = interactions['user_id'].map(user_id_map)
interactions['book_id'] = interactions['book_id'].map(book_id_map)

n_users = interactions['user_id'].nunique()
n_items = interactions['book_id'].nunique()
print(f'Users: {n_users} | Books: {n_items} | Interactions: {len(interactions)}')


## Step 3: Helper Functions

In [ ]:
def create_weighted_matrix(data, n_users, n_items, decay=0.03):
    """Rank-based time-decay weighted matrix."""
    mat = np.zeros((n_users, n_items))
    for _, ud in data.groupby('user_id'):
        ud = ud.sort_values('timestamp'); n = len(ud)
        w = np.array([(1 - decay) ** (n - 1 - i) for i in range(n)]); w /= w.max()
        for i, (_, row) in enumerate(ud.iterrows()):
            mat[int(row['user_id']), int(row['book_id'])] = w[i]
    return mat

def create_data_matrix(data, n_users, n_items):
    mat = np.zeros((n_users, n_items))
    mat[data['user_id'].values, data['book_id'].values] = 1
    return mat

def user_based_predict(mat, sim, eps=1e-9):
    return sim.dot(mat) / (np.abs(sim).sum(axis=1)[:, None] + eps)

def item_based_predict(mat, sim, eps=1e-9):
    return (sim.dot(mat.T) / (sim.sum(axis=1)[:, None] + eps)).T

def normalize(mat):
    lo, hi = mat.min(), mat.max()
    return (mat - lo) / (hi - lo + 1e-9)

def sln(x):
    """Sign-preserving log normalization — keeps the direction of CF scores."""
    return normalize(np.log1p(np.abs(x)) * np.sign(x))

def precision_recall_at_k(scores, gt, k=10):
    n = scores.shape[0]; total_p, total_r = 0.0, 0.0
    for u in range(n):
        true_items = np.where(gt[u] == 1)[0]
        top_k = np.argsort(scores[u])[-k:]
        hits  = np.isin(top_k, true_items).sum()
        total_p += hits / k
        total_r += hits / len(true_items) if len(true_items) > 0 else 0
    return total_p / n, total_r / n


## Step 4: Graph Random Walk with Restart

We build a **bipartite graph** connecting users and books. Starting from a user node, a virtual explorer randomly follows edges (user → borrowed books → other users who read those books → their books…) with a restart probability of **α = 0.7** — meaning at each step there's a 30% chance to teleport back to the starting user.

After 15 iterations, the steady-state probability on each book node reflects how strongly it is connected to that user through the network, uncovering **hidden paths** that direct CF misses.

In [ ]:
def build_graph(binary_mat, n_users, n_items, alpha=0.7, n_iter=15):
    R = csr_matrix(binary_mat)
    adj = bmat([[csr_matrix((n_users, n_users)), R],
                [R.T, csr_matrix((n_items, n_items))]], format='csr')
    rs = np.array(adj.sum(axis=1)).flatten(); rs[rs == 0] = 1
    T  = diags(1.0 / rs).dot(adj)
    n_total = n_users + n_items
    scores = np.zeros((n_users, n_items))
    for bs in range(0, n_users, 200):
        be = min(bs + 200, n_users)
        p  = np.zeros((n_total, be - bs))
        for i, u in enumerate(range(bs, be)): p[u, i] = 1.0
        r = p.copy()
        for _ in range(n_iter): r = alpha * T.dot(r) + (1 - alpha) * p
        scores[bs:be] = r[n_users:].T
    return normalize(scores)


## Step 5: Count-Weighted Content Profiles

An improvement over Model 5: instead of averaging all borrowed books equally, we weight each book's TF-IDF vector by **log(1 + borrow_count)**. If a user borrowed the same book 5 times, it contributes more to their taste profile than a book they read once.

In [ ]:
def build_content(tfidf_mat, train_df, mapping, count_dict, n_users, n_items, book_id_map, books_i):
    profiles = np.zeros((n_users, tfidf_mat.shape[1]), dtype=np.float32)
    user_books = train_df.groupby('user_id')['book_id'].apply(list).to_dict()
    for u in range(n_users):
        rows, wts = [], []
        for b in user_books.get(u, []):
            if b in mapping:
                wts.append(np.log1p(count_dict.get((u, b), 1)))
                rows.append(mapping[b])
        if not rows: continue
        w = np.array(wts, dtype=np.float32); w /= w.sum()
        profiles[u] = np.average(tfidf_mat[rows].toarray(), weights=w, axis=0)
    norms = np.linalg.norm(profiles, axis=1, keepdims=True); norms[norms == 0] = 1
    profiles /= norms
    all_sc = cosine_similarity(profiles, tfidf_mat)
    content = np.zeros((n_users, n_items), dtype=np.float32)
    for i, orig in enumerate(books_i):
        if orig in book_id_map: content[:, book_id_map[orig]] = all_sc[:, i]
    return normalize(np.log1p(content))


## Step 6: TF-IDF Preparation

In [ ]:
for col in ['Title', 'Author', 'Subjects', 'Publisher']: books[col] = books[col].fillna('')
books['text'] = (books['Title'] + ' ' + books['Author'] + ' ' + books['Author'] + ' ' +
                 books['Subjects'] + ' ' + books['Subjects'] + ' ' + books['Publisher'])
tfidf_mat = TfidfVectorizer(max_features=10000, strip_accents='unicode', min_df=2).fit_transform(books['text'])
mapping = {book_id_map[o]: i for i, o in enumerate(books['i']) if o in book_id_map}
books_i  = books['i'].values
print(f'TF-IDF matrix shape: {tfidf_mat.shape}')


## Step 7: 5-Fold Cross-Validation

Weights were tuned with **Optuna** (200 trials, 5-fold average as objective) and then validated on Kaggle:

```
Final = normalize(0.8 × normalize(0.75 × CF + 0.20 × Content + 0.05 × Pop)
                + 0.2 × Graph)
```

In [ ]:
interactions_s = interactions.sort_values(['user_id', 'timestamp']).copy()
interactions_s['fold'] = interactions_s.groupby('user_id')['timestamp'].transform(
    lambda x: pd.qcut(x.rank(method='first'), 5, labels=False))

precisions, recalls = [], []

for fold in range(5):
    print(f'Fold {fold+1}/5...')
    train = interactions_s[interactions_s['fold'] != fold]
    test  = interactions_s[interactions_s['fold'] == fold]

    test_mat = np.zeros((n_users, n_items))
    test_mat[test['user_id'].values, test['book_id'].values] = 1

    count_dict = {(r['user_id'], r['book_id']): r['c']
                  for _, r in train.groupby(['user_id', 'book_id']).size()
                  .reset_index(name='c').iterrows()}

    # CF with sign-preserving log normalization
    train_mat = create_weighted_matrix(train, n_users, n_items)
    cf = 0.45 * sln(user_based_predict(train_mat, cosine_similarity(train_mat))) + \
         0.55 * sln(item_based_predict(train_mat, cosine_similarity(train_mat.T)))

    # Count-weighted content
    content = build_content(tfidf_mat, train, mapping, count_dict,
                            n_users, n_items, book_id_map, books_i)

    # Popularity
    binary_mat = create_data_matrix(train, n_users, n_items)
    pop = normalize(np.log1p(binary_mat.sum(axis=0)))

    # Graph RWR
    print('  Building graph...')
    graph = build_graph(binary_mat, n_users, n_items)

    # Final hybrid (Optuna-tuned weights)
    current = normalize(0.75 * cf + 0.20 * content + 0.05 * pop)
    hybrid  = normalize(0.8 * current + 0.2 * graph)

    p, r = precision_recall_at_k(hybrid, test_mat)
    precisions.append(p); recalls.append(r)
    print(f'  Precision@10: {p:.4f} | Recall@10: {r:.4f}')


## Results

In [ ]:
print('=' * 45)
print('AVERAGE RESULTS (5-FOLD CV):')
print(f'  Precision@10: {np.mean(precisions):.4f} ± {np.std(precisions):.4f}')
print(f'  Recall@10:    {np.mean(recalls):.4f} ± {np.std(recalls):.4f}')
print('=' * 45)
for i, (p, r) in enumerate(zip(precisions, recalls)):
    print(f'  Fold {i+1}: P={p:.4f}  R={r:.4f}')


## Key Takeaways

- **Sign-preserving log (sln)** prevents extreme CF scores from dominating after normalization, giving a more balanced signal.
- **Graph RWR** adds ~+0.0005 by surfacing books that are 2–3 hops away in the borrowing network — impossible to find with direct CF.
- **Optuna** searches 200 weight combinations automatically, replacing manual grid search and finding a more optimal balance.
- This model achieved a Kaggle score of **0.1739** (Precision@10 on the held-out test set).